In [13]:
from dotenv import load_dotenv
import os

load_dotenv() 

True

In [14]:
!pip install -q langchain-openai langchain-core requests


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)




In [16]:
from langchain_core.tools import tool

@tool
def convert_factor(base_currency: str , target_currency: str) -> float :
    """Fetches the currency conversion factor between base_currency and target_currency"""
    import requests
    url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'
    response = requests.get(url)
    return response.json()

In [17]:
print(convert_factor.invoke({"base_currency": "USD", "target_currency": "INR"}))

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1785801601, 'time_last_update_utc': 'Tue, 04 Aug 2026 00:00:01 +0000', 'time_next_update_unix': 1785888001, 'time_next_update_utc': 'Wed, 05 Aug 2026 00:00:01 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 95.395}


In [38]:
@tool
def convert(base_currency_value: int,conversion_rate: float) -> float:
    """Given a conversion rate, converts base_currency_value into the target currency"""
    return base_currency_value*conversion_rate

In [39]:
print(convert.invoke({"base_currency_value":10 , "conversion_rate" : 95.5}))

955.0


In [40]:
llm_with_tools = llm.bind_tools([convert_factor, convert])

In [41]:
from langchain_core.messages import HumanMessage

In [42]:
message = [HumanMessage("what is convertion rate between usd to inr")]

response = llm_with_tools.invoke(message)
print(response.tool_calls[0])

{'name': 'convert_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'bee2e8cf-5db0-472f-bd98-891c008aa384', 'type': 'tool_call'}


In [43]:
message.append(response)
final = convert_factor.invoke(response.tool_calls[0])
message.append(final)

out = llm_with_tools.invoke(message)
print(out.content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 1.249230147s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

In [ ]:
message=[HumanMessage("what is the convertion rate for usd to inr , and convert 10 usd to inr")]

ai_message = llm_with_tools.invoke(message)
message.append(ai_message)

print(ai_message.tool_calls)

[{'name': 'convert_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'a7ca7a31-19ee-41ee-87d3-5c45d1a82f1c', 'type': 'tool_call'}]


In [ ]:
import json

conversion_rate = None

for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'convert_factor':
        result = convert_factor.invoke(tool_call)
        conversion_rate = json.loads(result.content)['conversion_rate']
        message.append(result)

    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate  # manually inject it
        result = convert.invoke(tool_call)
        message.append(result)

final = llm_with_tools.invoke(message)
print(final.content)